# Generating-Unit Outage Screen with Logistic Regression – Practice Skeleton

**Short name (GitHub):** `Energy_LogReg`  
**Pattern:** same L1-logistic lab as `Income_LogReg` / `Bet_LogReg`, rebuilt for an energy operations blotter.  
**Data:** `data/energy_units.csv` (12,000 synthetic generating-unit days).  
**Target:** `forced_outage` is `Yes` / `No` — full outage **or material derate** that day (the Yes rate is therefore higher than a pure GADS forced-outage rate).

Teaching file only. Not an operating reliability tool and not a real ISO/GADS extract.

Companion files: `Energy_LogReg_Solution.ipynb`, `Energy_LogReg_Reusable_Template.ipynb`, `Energy_LogReg_Cheatsheet.docx`, `Energy_LogReg_Project_Memo.docx`, `Energy_LogReg_Strategy_Guide.docx`, `Energy_LogReg_1Page_Summary_Report.docx`, `energy_logreg_flowchart.png`.

## Inline cheat-sheet (keep this cell visible)

See also **`Energy_LogReg_Cheatsheet.docx`**.

| Item | Formula / code |
|------|----------------|
| Load | `pd.read_csv("data/energy_units.csv")` |
| Class mix | `df.forced_outage.value_counts(normalize=True)` |
| Dummies | `X = pd.get_dummies(df[cols], drop_first=True).astype(float)` |
| Binary target | `y = np.where(df.forced_outage == "No", 0, 1)` |
| Split | `train_test_split(X, y, test_size=0.2, random_state=1)` |
| L1 LR | `LogisticRegression(C=0.05, penalty="l1", solver="liblinear")` |
| Confusion | `[[TN, FP], [FN, TP]]` with positive = Yes (event) |
| Accuracy | compare to the No baseline (~0.74) |
| Coef table | `pd.DataFrame({"var": cols, "coef": model.coef_[0]})` |
| ROC / AUC | `roc_curve(y, p); roc_auc_score(y, p)` |

**Signs you should recover:** `age_years` +, `is_peak_hour` +, `days_since_maint` +, hydro/solar/nuclear − versus the dropped **coal** reference. On the expanded card, `maint_overdue` is large and positive.

## 0. Packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    precision_score, recall_score, f1_score, roc_curve, roc_auc_score,
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

## 1. Load the unit-day blotter

### Task 1.1

Read `data/energy_units.csv`. Print `head()`, `shape`, and how many unique `unit_id` values there are. Repeated units mean the independence assumption is only approximate.

In [ ]:
# YOUR CODE HERE
df = None
print(df.head())
print(df.shape)
print("unique units", None)

## 2. EDA and assumptions

### Task 2.1 — class imbalance

Print counts and the mix. A screen that always says “No event” is already right about 74% of the time. Why will a board pack that only shows accuracy look better than the plant is?

In [ ]:
# YOUR CODE HERE
print(df.forced_outage.value_counts())
print(df.forced_outage.value_counts(normalize=True))

### Task 2.2 — dummy-encode the working feature set

```python
feature_cols = ["age_years", "ambient_c", "capacity_mw", "load_factor",
                "days_since_maint", "is_peak_hour", "fuel", "region"]
```

`get_dummies(..., drop_first=True)`, cast to `float`. Dropped fuel reference is **coal**; dropped region is **ERCOT**.

In [ ]:
feature_cols = [
    "age_years", "ambient_c", "capacity_mw", "load_factor",
    "days_since_maint", "is_peak_hour", "fuel", "region",
]
# YOUR CODE HERE
X = None
print(X.shape)
print(list(X.columns))

### Task 2.3 — correlation heatmap

In [ ]:
# YOUR CODE HERE
plt.figure(figsize=(11, 9))
plt.title("Feature correlation (dummy-encoded X)")
plt.tight_layout()
plt.show()

### Task 2.4 — scale check, then encode `y`

Print min / max / mean for `age_years`, `ambient_c`, `capacity_mw`, `load_factor`, `days_since_maint`.

`y = 1` when `forced_outage == "Yes"`.

In [ ]:
# YOUR CODE HERE
y = None
print("event rate", y.mean())

## 3. Fit the lesson model

### Task 3.1

`random_state=1`, `test_size=0.2`, `LogisticRegression(C=0.05, penalty="l1", solver="liblinear")`.

In [ ]:
# YOUR CODE HERE
x_train = x_test = y_train = y_test = None
log_reg = LogisticRegression(C=0.05, penalty="l1", solver="liblinear")
y_pred = None

### Task 3.2 — intercept and coefficients

In [ ]:
print("Model Parameters, Intercept:")
# YOUR CODE HERE
print("Model Parameters, Coeff:")
# YOUR CODE HERE

### Task 3.3 — confusion matrix and accuracy

Ballpark: accuracy near **0.78**, event recall near **0.22** at t = 0.5. The model is a conservative reserve flag, not a trip-wire that catches every derate.

In [ ]:
print("Confusion Matrix on test set:")
# YOUR CODE HERE
print("Accuracy Score on test set:")
# YOUR CODE HERE

## 4. Coefficient table and bar plot

In [ ]:
# YOUR CODE HERE
coef_df = None
print(coef_df)

In [ ]:
# YOUR CODE HERE
plt.figure(figsize=(10, 6))
plt.xticks(rotation=90)
plt.title("LR Coefficient Values (forced outage)")
plt.tight_layout()
plt.show()

## 5. ROC curve and AUC

A reliability screen that cannot rank is useless for day-ahead reserve. AUC near 0.75 is a *weak operational screen*, not a digital twin of the plant.

In [ ]:
# YOUR CODE HERE
y_pred_prob = None
roc_auc = None
print("ROC AUC score:", roc_auc)

## 6. Alternate code

### Task 6.1 — scaled L1 pipeline

In [ ]:
# YOUR CODE HERE
pipe = None

### Task 6.2 — L2 default vs L1 zeros

In [ ]:
# YOUR CODE HERE
log_l2 = None

### Task 6.3 — age-only card

Fit L1 on `age_years` alone. How much AUC do fuel and peak add?

In [ ]:
# YOUR CODE HERE


### Task 6.4 — threshold as reserve policy

Sweep t = 0.20, 0.30, 0.40, 0.50. Low t = hold more reserves (catch more events, more false alarms). High t = lean reserve stack.

In [ ]:
def predict_at(proba, t=0.5):
    # YOUR CODE HERE
    return None
for t in (0.20, 0.30, 0.40, 0.50):
    pass

## 7. More practice

### Task 7.1 — add `maint_overdue`, `humidity`, `operator`

Which new coefficient is largest? What happens to AUC and event-recall?

In [ ]:
# YOUR CODE HERE


### Task 7.2 — `class_weight="balanced"`

Report event-recall vs the default model. Accuracy will drop — that is a more conservative reserve posture.

In [ ]:
# YOUR CODE HERE


### Task 7.3 — fuel slice

On the lesson test fold, compute event-recall for coal-reference rows vs `fuel_nuclear==1` vs `fuel_wind==1`. A gap is a *fleet* diagnostic, not proof one technology is “unreliable” in the real interconnection.

In [ ]:
# YOUR CODE HERE


### Task 7.4 — which metric when?

One sentence each:

* ISO day-ahead reserve (missing an outage is costly)
* capex memo that should not cry wolf on healthy nuclear units
* board KPI that only wants “how often are we right” 

In [ ]:
iso_reserve = """..."""
capex_memo = """..."""
board_kpi = """..."""
print(iso_reserve); print(capex_memo); print(board_kpi)

## 8. Simulation (edit the boxed parameters)

Each replicate draws `N` unit-days with replacement, optionally flips a `NOISE` fraction of training labels (mis-coded GADS events), fits lesson L1, and scores at threshold `T`.

In [ ]:
# --- editable parameters ---
C = 0.05
N = 6000
N_REPS = 12
NOISE = 0.00
T = 0.50
SEED = 1
# ---------------------------
rng = np.random.default_rng(SEED)
rows = []
# YOUR CODE HERE
sim = pd.DataFrame(rows)
print(sim.describe().round(3) if len(sim) else "no rows yet")

## 9. Audience rewrite

Using Jočys and McMurrey, rewrite **one finding**: *unscaled L1 logistic regression on age, ambient temperature, capacity, load factor, days since maintenance, peak hour, fuel and region reaches test accuracy ≈ 0.78 and AUC ≈ 0.76, but event-recall at t = 0.5 is only ≈ 0.22.*

Four seats: **reliability engineer**, **plant-ops analyst**, **utility / ISO executive**, **ratepayer**.

In [ ]:
expert = """..."""
technician = """..."""
executive = """..."""
nonspecialist = """..."""
print("EXPERT\n", expert)
print("TECHNICIAN\n", technician)
print("EXECUTIVE\n", executive)
print("NONSPECIALIST\n", nonspecialist)

## 10. When this project is a good fit — and when it is not

In [ ]:
good_fit = []
not_a_fit = []
for row in good_fit: print(row)
print("--- not a fit ---")
for row in not_a_fit: print(row)

## 11. Done checklist

- [ ] 12,000-row blotter loaded; unique `unit_id` counted
- [ ] ~74 / 26 imbalance named against the No baseline
- [ ] 16-column dummy matrix, heatmap, scale note
- [ ] L1 model, CM, accuracy, sparse coefs, ROC
- [ ] Alternates + threshold-as-reserve-policy
- [ ] Maintenance card, balanced weights, fuel slice
- [ ] Simulation knobs
- [ ] Four-audience rewrite + good-fit list
- [ ] No claim that this is a live GADS model